In [1]:
print("hello")

hello


In [3]:
# Loading the SQL module
%load_ext sql

In [4]:
import os 
import socket

from dotenv import load_dotenv

load_dotenv('./src/env', override=True)

DBHOST = socket.gethostname()
DBPORT = os.getenv('DBPORT')
DBNAME = os.getenv('DBNAME')
DBUSER = os.getenv('DBUSER')
DBPASSWORD = os.getenv('DBPASSWORD')

connection_url = f'mysql+pymysql://{DBUSER}:{DBPASSWORD}@{DBHOST}:{DBPORT}/{DBNAME}'

%sql {connection_url}

Traceback (most recent call last):
  File "c:\WORK\Data Engineering Analysis Projects\sakila_sample_db\.venv\Lib\site-packages\sql\connection.py", line 45, in __init__
    engine = sqlalchemy.create_engine(
        connect_str, connect_args=connect_args
    )
  File "<string>", line 2, in create_engine
  File "c:\WORK\Data Engineering Analysis Projects\sakila_sample_db\.venv\Lib\site-packages\sqlalchemy\util\deprecations.py", line 281, in warned
    return fn(*args, **kwargs)  # type: ignore[no-any-return]
  File "c:\WORK\Data Engineering Analysis Projects\sakila_sample_db\.venv\Lib\site-packages\sqlalchemy\engine\create.py", line 564, in create_engine
    u = _url.make_url(url)
  File "c:\WORK\Data Engineering Analysis Projects\sakila_sample_db\.venv\Lib\site-packages\sqlalchemy\engine\url.py", line 856, in make_url
    return _parse_url(name_or_url)
  File "c:\WORK\Data Engineering Analysis Projects\sakila_sample_db\.venv\Lib\site-packages\sqlalchemy\engine\url.py", line 917, in _par

In [ ]:
#Write an SQL query to get the total amount obtained from the renting of Travel, Family, and Children films during the months of June and July of 2005. Present the results grouped and ordered by store id and category name.

In [ ]:
%%sql
SELECT
    fact_rental.store_id,
    dim_category.name AS category_name,
    sum(fact_rental.amount) AS total_amount
FROM
    fact_rental
    INNER JOIN dim_category ON fact_rental.category_id = dim_category.category_id
WHERE
    dim_category.name IN ('Travel', 'Family', 'Children')
    AND fact_rental.rental_date BETWEEN '2005-06-01' AND '2005-08-01'
GROUP BY
    store_id,
    dim_category.name
ORDER BY
    fact_rental.store_id;

In [ ]:
#CTE practice 1 : You can start by creating a query to extract the `category_id` and `film_id` from the `fact_rental`. Use [`DISTINCT`](https://www.w3schools.com/sql/sql_distinct.asp) and `LIMIT`. To compare your results with the expected output, you can order by the `category_id` and `film_id`.

%%sql
SELECT DISTINCT 
    category_id, 
    film_id
FROM
    fact_rental
ORDER BY
    category_id,
    film_id
LIMIT 10

#CTE practice 2: Let's practice writing Common Table Expressions (CTEs). Use optional exercise 2.1 without the `LIMIT` and `ORDER BY` statements as the code base to create a temporary result table named `film_category`. From the CTE `film_category`, select the `category_id` and aggregate the number of films in each category using the `COUNT()` function. Name the output column as `films`. Perform grouping by `category_id`. To check your result against the expected output, add the `ORDER BY` column `category_id` and `LIMIT` for the top 10 rows

%%sql
WITH film_category AS (
    SELECT DISTINCT 
        category_id,
        film_id
    FROM
        fact_rental
)
SELECT
    category_id,
    count(*) AS films
FROM
    film_category
GROUP BY
    category_id
ORDER BY
    category_id
LIMIT 10;

#CTE practice 3: In this exercise, create a query with 2 CTEs. The first CTE uses the code for `film_category` as in optional exercise 2.2, and the second CTE is based on the code for the `SELECT` part of the query for optional exercise 2.2, naming the second CTE as `film_category_count`. Here is the scheme showing how to use two CTEs in the query:
%%sql
WITH film_category AS (
    SELECT DISTINCT 
        category_id,
        film_id
    FROM
        fact_rental
),
film_category_count AS (
    SELECT
        category_id,
        count(*) AS films
    FROM
        film_category
    GROUP BY
        category_id
    ORDER BY
        category_id
)
SELECT
    avg(films) AS avarage_by_category
FROM
    film_category_count;


In [7]:
#Write an SQL query using a CTE to get the average number of films per category. Then, calculate the average rounded down and rounded up.

In [ ]:
%%sql
WITH film_category AS (
    SELECT DISTINCT 
        category_id,
        film_id
    FROM
        fact_rental
),
film_category_count AS (
    SELECT
        category_id,
        count(*) AS films
    FROM
        film_category
    GROUP BY
        category_id
    ORDER BY
        category_id
),
film_average_by_category AS (
    SELECT
        avg(films) AS average
    FROM
        film_category_count
)
SELECT
    average AS average,
    FLOOR(average) AS average_down,
    CEIL(average) AS average_up
FROM
    film_average_by_category;

In [ ]:
#Write an SQL query using a subquery to get the film categories that have the number of films above the average rounded up calculated in the previous exercise.

In [ ]:
%%sql
WITH film_category AS (
    SELECT DISTINCT 
        category_id,
        film_id
    FROM
        fact_rental
),
film_category_count AS (
    SELECT
        category_id,
        count(*) AS films
    FROM
        film_category
    GROUP BY
        category_id
    ORDER BY
        category_id
)
SELECT
    *
FROM
    film_category_count
WHERE
    films > (
        SELECT
            CEIL(AVG(films))
        FROM
            film_category_count
    );